# Finance Tracker - Local Testing
Test the core pipeline (parse → save → confirm) without the WhatsApp webhook layer.

In [ ]:
import sys
import uuid
from pathlib import Path
from dotenv import load_dotenv

# Load .env before importing app modules
load_dotenv(Path.cwd().parent / ".env")

# Add src/ to path so we can import our modules
sys.path.insert(0, str(Path.cwd().parent / "src"))

## Test LLM Parsing
Send a text string to the OpenAI parser and inspect the structured output.

In [ ]:
from parser import parse_expense

expense = parse_expense("me gasté 50 hoy")
expense

In [ ]:
# Test with payment method
expense_card = parse_expense("mercado 120000 tarjeta")
expense_card

In [ ]:
# Test with merchant
expense_merchant = parse_expense("uber 14000")
expense_merchant

## Test Database Persistence
Save a message and an expense to Neon PostgreSQL, then verify.

In [ ]:
from database import save_message, save_expense, get_connection

# Save a test message
test_msg_id = f"test-{uuid.uuid4().hex[:8]}"
message_id = save_message(test_msg_id, "573001234567", "almuerzo 32000")
print(f"Saved message with id: {message_id}")

# Save the parsed expense linked to the message
expense_id = save_expense(message_id, expense)
print(f"Saved expense with id: {expense_id}")

In [ ]:
# Verify: query the data back
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT m.raw_text, e.amount, e.category, e.expense_date, e.confidence
            FROM expenses e
            JOIN messages m ON m.id = e.message_id
            ORDER BY e.created_at DESC
            LIMIT 5;
        """)
        rows = cur.fetchall()
        for row in rows:
            print(row)

## Test Confirmation Message
Verify the WhatsApp confirmation text is formatted correctly.

In [ ]:
from whatsapp import format_confirmation

print(format_confirmation(expense))
print(format_confirmation(expense_card))
print(format_confirmation(expense_merchant))

## Full Pipeline Test
Run the complete flow: parse → save → format confirmation.

In [ ]:
def test_full_pipeline(text: str):
    """Run the full expense pipeline without WhatsApp."""
    # Parse
    exp = parse_expense(text)
    
    # Save
    msg_id = save_message(f"test-{uuid.uuid4().hex[:8]}", "573001234567", text)
    save_expense(msg_id, exp)
    
    # Confirm
    confirmation = format_confirmation(exp)
    print(f"Input: {text}")
    print(f"Output: {confirmation}")
    print(f"Parsed: {exp}")
    print("---")


test_full_pipeline("uber 14 lukas")